# Colab task: GPU worker with **Ollama + Llama**

This notebook:

1. Installs and starts **Ollama** on Colab (GPU).
2. Pulls **`llama3.2:1b`** (change `OLLAMA_MODEL` for other tags).
3. Runs the project **worker** with **`LLM_BACKEND=ollama`** → calls `http://127.0.0.1:11434/api/chat` (see `llm/inference.py`).

The worker still uses **FAISS RAG** locally; only generation goes through Ollama.

**Broker**: **`REDIS_URL`** must reach the same Redis your control plane uses. Run the API from **`03_Master_Control_Plane.ipynb`**, VPS, or any host pointing workers at that Redis endpoint.

## 1) GPU check

In [ ]:
!nvidia-smi -L

## 2) Project root on Colab

- **Git**: clone into `/content/dist`.
- **Upload**: unzip the project under `/content/` and set `PROJECT_ROOT`.

In [ ]:
import os

# !git clone https://github.com/YOUR_ORG/YOUR_REPO.git /content/dist

PROJECT_ROOT = "/content/dist"
os.chdir(PROJECT_ROOT)
print("cwd:", os.getcwd())
assert os.path.isfile("main.py"), "Set PROJECT_ROOT so main.py exists"

## 3) Install and start **Ollama** (GPU)

Uses the official install script, then runs `ollama serve` in the background. Logs: `/content/ollama.log`.

In [ ]:
import os
import subprocess
import time

!sudo apt-get update -qq && sudo apt-get install -y -qq pciutils zstd

!curl -fsSL https://ollama.com/install.sh | sh


def start_ollama_server() -> None:
    out = open("/content/ollama.log", "w")
    err = open("/content/ollama_error.log", "w")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=out,
        stderr=err,
        start_new_session=True,
    )
    print("Launched ollama serve (see /content/ollama.log)")


start_ollama_server()
time.sleep(10)
!ollama --version

## 4) Pull **Llama 3.2** (Ollama)

Default model tag: **`llama3.2:1b`** (~small download). Edit the tag to use another Ollama image.

In [ ]:
import os
import subprocess
import sys

OLLAMA_PULL_MODEL = os.environ.get("OLLAMA_MODEL", "llama3.2:1b")
print("Pulling:", OLLAMA_PULL_MODEL)

proc = subprocess.Popen(
    ["ollama", "pull", OLLAMA_PULL_MODEL],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
    sys.stdout.flush()
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"ollama pull failed with code {proc.returncode}")

print("Model ready.")

## 5) Quick Ollama smoke test (optional)

In [ ]:
import os
import requests

model = os.environ.get("OLLAMA_MODEL", "llama3.2:1b")
r = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": model,
        "prompt": "What is the capital of Egypt? Answer in one word.",
        "stream": False,
        "options": {"temperature": 0, "num_predict": 32},
    },
    timeout=120,
)
r.raise_for_status()
data = r.json()
print(data.get("response", data).strip())

## 6) Python packages (worker + RAG, **no** HF/bitsandbytes)

Uses **`requirements-ollama-worker.txt`** at the project root. Colab usually has **torch**; `sentence-transformers` will reuse it.

In [ ]:
%pip install -q -r requirements-ollama-worker.txt

## 7) Environment (Redis + **Ollama** backend)

- **`LLM_BACKEND=ollama`** → `llm/inference.py` uses **`OllamaEngine`** (`/api/chat`).
- **`OLLAMA_MODEL`**: must match the pulled tag (default `llama3.2:1b`).
- **`RAY_DISABLE=1`** recommended on Colab when not using Ray scheduling (plain worker loop).

In [ ]:
import os
from getpass import getpass

os.chdir(PROJECT_ROOT)

os.environ["REDIS_URL"] = os.environ.get("REDIS_URL") or getpass("REDIS_URL: ")

os.environ["LLM_BACKEND"] = "ollama"
os.environ.setdefault("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
os.environ.setdefault("OLLAMA_MODEL", "llama3.2:1b")
os.environ.setdefault("RAG_TOP_K", "4")
os.environ.setdefault("RAY_DISABLE", "1")
os.environ.setdefault("LLM_MAX_NEW_TOKENS", "256")
os.environ.setdefault("LLM_TEMPERATURE", "0.7")

# HF not needed for Ollama path
# os.environ["WORKER_ID"] = "colab-worker-01"

## 8) Warm up **FAISS** RAG (optional)

For corpus / index tuning, run **`04_RAG_Module.ipynb`** first and reuse the same **`RAG_INDEX_DIR`**.

In [ ]:
import os
import sys

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from rag.retriever import FaissRetriever

r = FaissRetriever()
r.ensure_loaded()
print("chunks:", len(r._chunks))
print(r.retrieve_context("What is Redis Streams?", k=2))

## 9) Start the **worker** (blocks)

Consumes Redis Stream tasks, runs RAG + **Ollama Llama**, writes results, **XACK**.

Stop with Colab’s stop button.

In [ ]:
import os
import sys

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from workers.gpu_worker import main as worker_main

worker_main()